In [ ]:
# ==============================================================================
#  STATISTICAL TESTS — append Wilcoxon + McNemar sheets to the TEST XLSX
# ==============================================================================

from scipy.stats import wilcoxon
from openpyxl import load_workbook


DEFAULT_MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"
SEED = 62
N_FOLDS = 5
FAR_TARGET = 0.01

# The winning (rule, p) per system, as selected on TRAIN by
# evaluate_pdep_train_5fold.py (seed=62, p step=0.05). Override with
# --configs-csv to load a different set instead of editing this table.
BEST_CONFIGS: Dict[str, Tuple[str, float]] = {
    "S1": ("R5p",        0.4),
    "S2": ("Werners1",   0.45),
    "S3": ("R5p",        0.25),
    "S4": ("R5p",        0.30),
    "S5": ("Luhandjula", 0.45),
    "S6": ("Werners2",   0.45),
    "S7": ("Mizumoto2",  0.75),
    "S8": ("Luhandjula", 0.8),
}


# Fusion system → unimodal baselines to compare against
BASELINE_MAP: Dict[str, List[str]] = {
    "S1": ["Face_C", "Face_G"],
    "S2": ["Face_C", "Finger_LI"],
    "S3": ["Face_C", "Finger_RI"],
    "S4": ["Face_G", "Finger_LI"],
    "S5": ["Face_G", "Finger_RI"],
    "S6": ["Finger_LI", "Finger_RI"],
    "S7": ["Face_C", "Finger_LI", "Finger_RI"],
    "S8": ["Face_G", "Finger_LI", "Finger_RI"],
}


def _rank1_binary(S: np.ndarray, far_target: float, invalid_value: float) -> np.ndarray:
    """
    Per-probe Rank-1 success at FAR ≈ far_target.
    True = genuine is row-max AND genuine score ≥ impostor threshold at far_target.
    """
    S = np.asarray(S, dtype=float)
    n = S.shape[0]
    # Treat invalid_value as missing
    invalid = (S == invalid_value) if not np.isnan(invalid_value) else np.isnan(S)
    S_work = S.copy()
    S_work[invalid] = np.nan

    genuine = np.diag(S_work).copy()
    mask = ~np.eye(n, dtype=bool)
    impostor = S_work[mask]
    impostor = impostor[~np.isnan(impostor)]
    if len(impostor) < 10:
        return np.zeros(n, dtype=bool)

    sorted_imp = np.sort(impostor)[::-1]
    k = int(np.floor(far_target * len(sorted_imp)))
    k = min(max(k, 0), len(sorted_imp) - 1)
    thresh = sorted_imp[k]

    S_fill = np.where(np.isnan(S_work), -np.inf, S_work)
    row_max = np.max(S_fill, axis=1)
    valid = ~np.isnan(genuine)
    return valid & (genuine >= row_max - 1e-12) & (genuine >= thresh)


def _mcnemar_from_binary(b_ok: np.ndarray, f_ok: np.ndarray) -> Tuple[int, int, int, int, float, float]:
    """
    Build 2x2 table from paired boolean vectors; return a,b,c,d, chi2, p-value.
    Uses exact binomial test when b+c is small, else chi2 with continuity correction.
    """
    from scipy.stats import binomtest

    both_ok = int(np.sum(b_ok & f_ok))
    b_only = int(np.sum(b_ok & ~f_ok))   # baseline correct, fusion wrong
    f_only = int(np.sum(~b_ok & f_ok))   # baseline wrong, fusion correct
    both_bad = int(np.sum(~b_ok & ~f_ok))
    n_disc = b_only + f_only
    if n_disc == 0:
        return both_ok, b_only, f_only, both_bad, 0.0, 1.0
    # Exact McNemar (binomial test on discordant pairs)
    p_exact = binomtest(f_only, n_disc, 0.5).pvalue
    # Chi2 with continuity correction (for reporting)
    chi2 = ((abs(b_only - f_only) - 1) ** 2) / n_disc if n_disc else 0.0
    return both_ok, b_only, f_only, both_bad, chi2, float(p_exact)


def DoStats(mat_file: str,
            seed: int,
            n_folds: int,
            best_configs: Dict[str, Tuple[str, float]],
            report: pd.DataFrame,
            xlsx_path: Path,
            far_target: float = FAR_TARGET) -> Path:
    """
    Recompute baselines and fusion Rank-1 decisions on the same TEST folds
    (same seed, same TRAIN-only normalization), then append:

      - Wilcoxon   : paired tests on 5 fold-wise EER / Rank-1
      - McNemar    : probe-level Rank-1 @ FAR=1%
      - Baseline_PerFold : baseline metrics per fold

    Call after write_report(...).
    """
    from scipy.io import loadmat

    print("\n[DoStats] Computing baselines + paired tests ...")
    data_cfg = DataConfig(mat_file=mat_file)
    split_cfg = SplitConfig(n_folds=n_folds, seed=seed, shuffle=True)

    contents = loadmat(mat_file)
    raw_matrices = {
        modality: np.asarray(contents[key], dtype=float)
        for modality, key in data_cfg.matrix_keys.items()
    }
    n_subjects = next(iter(raw_matrices.values())).shape[0]
    folds = make_folds(n_subjects, split_cfg)
    invalid = data_cfg.invalid_value

    # modality names from DataConfig keys
    modalities = list(data_cfg.matrix_keys.keys())

    # Collectors
    baseline_fold = {m: {"eer": [], "r1": []} for m in modalities}
    fusion_fold_eer = {s: [] for s in best_configs}
    fusion_fold_r1  = {s: [] for s in best_configs}
    # Pooled binary Rank-1 vectors (concatenate over folds)
    baseline_bin = {m: [] for m in modalities}
    fusion_bin   = {s: [] for s in best_configs}

    for fold_idx in range(n_folds):
        test_ids = folds[fold_idx]
        train_ids = train_indices_for_fold(folds, fold_idx, n_subjects)

        normalized_test: Dict[str, np.ndarray] = {}
        for modality, matrix in raw_matrices.items():
            train_block = submatrix(matrix, train_ids)
            test_block = submatrix(matrix, test_ids)
            normalizer = make_normalizer(data_cfg, label=modality)
            normalizer.fit(train_block)
            normalized_test[modality] = normalizer.transform(test_block)

        # --- Baselines ---
        for m in modalities:
            S = normalized_test[m]
            genuine, impostor = split_genuine_impostor(S, invalid)
            eer = compute_eer(genuine, impostor) * 100.0
            r1  = compute_rank1_at_far(S, far_target, invalid) * 100.0
            baseline_fold[m]["eer"].append(eer)
            baseline_fold[m]["r1"].append(r1)
            baseline_bin[m].append(_rank1_binary(S, far_target, invalid))

        # --- Fusion systems (same fixed configs) ---
        for system_name, (rule, p) in best_configs.items():
            system = SYSTEM_DEFINITIONS[system_name]
            fused = build_system_matrix(
                system, normalized_test, rule, p, invalid
            )
            genuine, impostor = split_genuine_impostor(fused, invalid)
            eer = compute_eer(genuine, impostor) * 100.0
            r1  = compute_rank1_at_far(fused, far_target, invalid) * 100.0
            fusion_fold_eer[system_name].append(eer)
            fusion_fold_r1[system_name].append(r1)
            fusion_bin[system_name].append(_rank1_binary(fused, far_target, invalid))

        print(f"  [DoStats] Fold {fold_idx + 1}/{n_folds} done.")

    # Concatenate binary vectors across folds
    for m in modalities:
        baseline_bin[m] = np.concatenate(baseline_bin[m])
    for s in best_configs:
        fusion_bin[s] = np.concatenate(fusion_bin[s])

    # ------------------------------------------------------------------
    # Write sheets into existing workbook
    # ------------------------------------------------------------------
    wb = load_workbook(xlsx_path)
    header_fill = PatternFill("solid", fgColor="B45309")
    header_font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    cell_font = Font(name="Arial", size=10)
    border = Border(*(Side(style="thin", color="D9D9D9") for _ in range(4)))

    def _style_header(ws, row, n_cols):
        for c in range(1, n_cols + 1):
            cell = ws.cell(row=row, column=c)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", wrap_text=True)

    # ----- Baseline_PerFold -----
    ws_b = wb.create_sheet("Baseline_PerFold")
    ws_b["A1"] = "Unimodal baselines on the same TEST folds (TRAIN-fit normalization)"
    hdr = ["Modality", "Fold", "EER (%)", "Rank-1 @ FAR=1%"]
    for c, h in enumerate(hdr, 1):
        ws_b.cell(row=3, column=c, value=h)
    _style_header(ws_b, 3, len(hdr))
    r = 4
    for m in modalities:
        for f in range(n_folds):
            ws_b.cell(row=r, column=1, value=m).font = cell_font
            ws_b.cell(row=r, column=2, value=f + 1).font = cell_font
            ws_b.cell(row=r, column=3, value=round(baseline_fold[m]["eer"][f], 4)).font = cell_font
            ws_b.cell(row=r, column=4, value=round(baseline_fold[m]["r1"][f], 4)).font = cell_font
            r += 1
    for col in range(1, 5):
        ws_b.column_dimensions[get_column_letter(col)].width = 18

    # ----- Wilcoxon -----
    ws_w = wb.create_sheet("Wilcoxon")
    ws_w["A1"] = (
        "Paired Wilcoxon signed-rank test on the 5 fold-wise values "
        "(fusion vs baseline). H0: median difference = 0. "
        "Two-sided. With only 5 pairs, p-values are coarse; report exact p."
    )
    hdr = [
        "System", "Baseline", "Metric",
        "Fusion_mean", "Baseline_mean", "Delta (F-B)",
        "Wilcoxon_stat", "p_value", "Significant_0.05",
    ]
    for c, h in enumerate(hdr, 1):
        ws_w.cell(row=3, column=c, value=h)
    _style_header(ws_w, 3, len(hdr))

    r = 4
    for sid, baselines in BASELINE_MAP.items():
        if sid not in best_configs:
            continue
        for bl in baselines:
            if bl not in modalities:
                continue
            for metric, f_vals, b_vals in [
                ("EER", fusion_fold_eer[sid], baseline_fold[bl]["eer"]),
                ("Rank-1", fusion_fold_r1[sid], baseline_fold[bl]["r1"]),
            ]:
                f_arr = np.asarray(f_vals, dtype=float)
                b_arr = np.asarray(b_vals, dtype=float)
                # Wilcoxon needs non-zero differences; handle all-ties
                diff = f_arr - b_arr
                if np.allclose(diff, 0):
                    stat, pval = 0.0, 1.0
                else:
                    try:
                        res = wilcoxon(f_arr, b_arr, zero_method="wilcox", alternative="two-sided")
                        stat, pval = float(res.statistic), float(res.pvalue)
                    except ValueError:
                        stat, pval = np.nan, 1.0
                delta = float(np.mean(f_arr) - np.mean(b_arr))
                row_vals = [
                    sid, bl, metric,
                    round(float(np.mean(f_arr)), 4),
                    round(float(np.mean(b_arr)), 4),
                    round(delta, 4),
                    round(stat, 4) if not np.isnan(stat) else None,
                    round(pval, 6),
                    "YES" if pval < 0.05 else "no",
                ]
                for c, v in enumerate(row_vals, 1):
                    ws_w.cell(row=r, column=c, value=v).font = cell_font
                r += 1
    for col in range(1, len(hdr) + 1):
        ws_w.column_dimensions[get_column_letter(col)].width = 14

    # ----- McNemar -----
    ws_m = wb.create_sheet("McNemar")
    ws_m["A1"] = (
        "McNemar test on Rank-1 @ FAR=1% decisions (probe-level, all TEST folds pooled). "
        "Table: b = baseline correct & fusion wrong; c = baseline wrong & fusion correct. "
        "p-value = exact binomial test on discordant pairs."
    )
    hdr = [
        "System", "Baseline",
        "Both_correct", "B_only (b)", "F_only (c)", "Both_wrong",
        "n_discordant", "chi2_corr", "p_exact", "Significant_0.05",
        "Fusion_better",  # c > b
    ]
    for c, h in enumerate(hdr, 1):
        ws_m.cell(row=3, column=c, value=h)
    _style_header(ws_m, 3, len(hdr))

    r = 4
    for sid, baselines in BASELINE_MAP.items():
        if sid not in best_configs:
            continue
        f_vec = fusion_bin[sid]
        for bl in baselines:
            if bl not in modalities:
                continue
            b_vec = baseline_bin[bl]
            # Align lengths (should already match: same test probes per fold)
            n = min(len(b_vec), len(f_vec))
            a, b, c_cnt, d, chi2, pval = _mcnemar_from_binary(b_vec[:n], f_vec[:n])
            row_vals = [
                sid, bl, a, b, c_cnt, d,
                b + c_cnt, round(chi2, 4), round(pval, 6),
                "YES" if pval < 0.05 else "no",
                "YES" if c_cnt > b else ("tie" if c_cnt == b else "no"),
            ]
            for col, v in enumerate(row_vals, 1):
                ws_m.cell(row=r, column=col, value=v).font = cell_font
            r += 1
    for col in range(1, len(hdr) + 1):
        ws_m.column_dimensions[get_column_letter(col)].width = 14

    wb.save(xlsx_path)
    print(f"[DoStats] Sheets added → {xlsx_path}")
    return xlsx_path

if __name__ == "__main__":
    print("-" * 48)
    best_configs = BEST_CONFIGS

    print(f"Applying {len(best_configs)} fixed (rule, p) configs to the TEST folds "
          f"(seed={SEED}, {N_FOLDS} folds) -- no sweep.")
    for system, (rule, p) in best_configs.items():
        print(f"  {system}: {rule} (p={p:.2f})")

    report = evaluate_best_configs_on_test(
        DEFAULT_MAT_FILE, SEED, N_FOLDS, BEST_CONFIGS, FAR_TARGET
    )

    output_path = write_report(
        report, SEED, N_FOLDS, BEST_CONFIGS, Path("./Results")
    )
    print(f"\nReport written to: {output_path}")
    print(report.to_string(index=False))

    # --- statistical tests ---
    DoStats(
        mat_file=DEFAULT_MAT_FILE,
        seed=SEED,
        n_folds=N_FOLDS,
        best_configs=BEST_CONFIGS,
        report=report,
        xlsx_path=output_path,
        far_target=FAR_TARGET,
    )

    print("-" * 48)
    print("done\t" * 3)
    print("-" * 48)